In [ ]:
from pathlib import Path

input_file = Path('../data/raw/yelp_academic_dataset_review.json')
output_dir = Path('../data/processed')
output_prefix = 'split_file'
number_of_files = 10

output_dir.mkdir(parents=True, exist_ok=True)

if not input_file.exists():
    raise FileNotFoundError(f'Input file not found: {input_file.resolve()}')


In [ ]:
# Count the JSON lines before splitting
with input_file.open('r', encoding='utf-8') as file:
    total_lines = sum(1 for _ in file)

base_lines = total_lines // number_of_files
extra_lines = total_lines % number_of_files

print(f'Total lines: {total_lines:,}')
print(f'Output files: {number_of_files}')


In [ ]:
# Split the source file without dropping remaining rows
written_counts = []

with input_file.open('r', encoding='utf-8') as source:
    for file_number in range(1, number_of_files + 1):
        rows_for_file = base_lines + (1 if file_number <= extra_lines else 0)
        output_file = output_dir / f'{output_prefix}_{file_number:02d}.json'
        written = 0

        with output_file.open('w', encoding='utf-8') as target:
            for _ in range(rows_for_file):
                line = source.readline()
                if not line:
                    break
                target.write(line)
                written += 1

        written_counts.append(written)
        print(f'{output_file.name}: {written:,} rows')


In [ ]:
# Validate that every source row was written
total_written = sum(written_counts)

if total_written != total_lines:
    raise ValueError(f'Row mismatch: source={total_lines:,}, written={total_written:,}')

print(f'Validation passed. {total_written:,} rows were written.')
